# Heart Disease Prediction with Naive Bayes

**Predicting heart disease presence using Gaussian Naive Bayes classification**

---

| Detail | Value |
|--------|-------|
| **Dataset** | UCI Heart Disease Dataset |
| **Techniques** | Gaussian Naive Bayes, RobustScaler, LabelEncoder, Confusion Matrix |
| **Author** | Ahmed Alnahrawy |



## 1. Setup & Data Loading

Import libraries and load the UCI Heart Disease dataset.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv(                   '../data/heart_disease_uci.csv')
df.head()


## 2. Data Exploration

Inspect features, missing values, and target variable distribution.


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()


## 3. Preprocessing & Feature Engineering

Encode categorical features, handle missing data, and scale numerics.


In [ ]:
import numpy as np
from sklearn.impute import KNNImputer

df['chol'] = df['chol'].replace(0, np.nan)
df['trestbps'] = df['trestbps'].replace(0, np.nan)

df.drop(columns=['id', 'dataset', 'ca'], inplace=True)

df['thal'] = df['thal'].fillna('Unknown')
df['slope'] = df['slope'].fillna('Unknown')

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).drop(columns=['num']).columns
cat_cols = df.select_dtypes(include=['object', 'bool']).columns
imputer = KNNImputer(n_neighbors=5)
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum()

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.head()

In [ ]:
df['num_binary'] = (df['num'] > 0).astype(int)

sns.countplot(x=df['num_binary'])
plt.show()

In [ ]:
from sklearn.preprocessing import RobustScaler

X = df.drop(columns=['num', 'num_binary'])
y_bin = df['num_binary']

X_train, X_test, y_train_bin, y_test_bin = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Naive Bayes Classification

Train Gaussian Naive Bayes model and evaluate with confusion matrix and classification report.


In [ ]:
from sklearn.metrics import confusion_matrix

nb_bin = GaussianNB()
nb_bin.fit(X_train_scaled, y_train_bin)
y_pred_bin = nb_bin.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test_bin, y_pred_bin))
print(classification_report(y_test_bin, y_pred_bin))

cm = confusion_matrix(y_test_bin, y_pred_bin)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Binary Classification")
plt.show()

In [ ]:
df_disease = df[df['num'] > 0]
X_multi = df_disease.drop(columns=['num', 'num_binary'])
y_multi = df_disease['num']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi)

scaler_m = RobustScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled = scaler_m.transform(X_test_m)

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier

selector_m = SelectKBest(score_func=f_classif, k=10)
X_train_m_selected = selector_m.fit_transform(X_train_m_scaled, y_train_m)
X_test_m_selected = selector_m.transform(X_test_m_scaled)

knn_multi = KNeighborsClassifier(weights='distance', n_neighbors=5)
knn_multi.fit(X_train_m_selected, y_train_m)
y_pred_m = knn_multi.predict(X_test_m_selected)

print("Multi-Class Classification")
print( accuracy_score(y_test_m, y_pred_m))
print(classification_report(y_test_m, y_pred_m))

cm_m = confusion_matrix(y_test_m, y_pred_m)
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Oranges')
plt.title("Multi-Class Classification")
plt.show()

In [ ]:
y_pred_final = []

for i in range(len(y_pred_bin)):
    pred_bin = y_pred_bin[i]
    if pred_bin == 0:
        y_pred_final.append(0)
    else:
        instance = X_test.iloc[[i]] 
        instance_scaled = scaler_m.transform(instance)
        instance_selected = selector_m.transform(instance_scaled)
        pred_multi = knn_multi.predict(instance_selected)[0]
        y_pred_final.append(pred_multi)

y_test_original = df.loc[X_test.index, 'num']
print( accuracy_score(y_test_original, y_pred_final))
print(classification_report(y_test_original, y_pred_final))

cm_final = confusion_matrix(y_test_original, y_pred_final)
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Greens')
plt.title("Full Pipeline")
plt.show()

In [ ]:
acc_stage1 = accuracy_score(y_test_bin, y_pred_bin)
acc_stage2 = accuracy_score(y_test_m, y_pred_m)
acc_final = accuracy_score(y_test_original, y_pred_final)

models = ['Stage 1 (Binary NB)', 'Stage 2 (KNN)', 'Full Pipeline']
accuracies = [acc_stage1, acc_stage2, acc_final]
bars = plt.bar(models, accuracies, color=['cyan', 'orange', 'green'])
plt.bar_label(bars, fmt='%.2f')
plt.ylabel('Accuracy')
plt.show()


---

## Summary

This notebook demonstrated gaussian naive bayes, robustscaler, labelencoder, confusion matrix techniques applied to the UCI Heart Disease Dataset.

**Author:** Ahmed Alnahrawy | [GitHub](https://github.com/Ahmed-Na7rawy)
